In [ ]:
# Setup
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data import load_data, RAW_DIR, TRUE_ELASTICITY
from utils import save_fig, plot_price_vs_quantity

pd.set_option('display.max_columns', None); pd.set_option('display.width', 120)
sns.set_style('whitegrid')
print(f"Raw data dir: {RAW_DIR}")
print(f"Ground-truth elasticity: {TRUE_ELASTICITY}")

In [ ]:
# 1. Load data
df = load_data()
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

In [ ]:
# 2. Summary stats by product tier
print(df.groupby('product_tier').agg(
    n=('transaction_id', 'count'),
    avg_price=('price', 'mean'),
    avg_qty=('quantity', 'mean'),
    total_revenue=('revenue', 'sum'),
    total_profit=('profit', 'sum'),
).round(2))

In [ ]:
# 3. Price vs quantity scatter (log-log)
plot_price_vs_quantity(df)
save_fig('01_price_vs_quantity.png')
plt.show()

In [ ]:
# 4. Revenue and profit by product tier
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
df.groupby('product_tier')['revenue'].sum().plot(kind='bar', ax=axes[0], color='#216b91')
axes[0].set_title('Total revenue by tier'); axes[0].set_ylabel('Revenue ($)')
df.groupby('product_tier')['profit'].sum().plot(kind='bar', ax=axes[1], color='#b05b3e')
axes[1].set_title('Total profit by tier'); axes[1].set_ylabel('Profit ($)')
save_fig('02_revenue_profit_by_tier.png')
plt.show()

In [ ]:
# 5. Promo effect
print("Average quantity by promo flag and tier:")
print(df.groupby(['product_tier', 'promo'])['quantity'].mean().round(1))

In [ ]:
# 6. Save cleaned copy
from data import save_processed
out = save_processed(df, name='transactions_clean.csv')
print(f"Saved → {out}")